# Nhóm 08 – Tối ưu siêu tham số MLP trên Fashion-MNIST (HPO)
**Notebook 01 (SV1): Dữ liệu + Baseline + Config dùng chung**

> Trước khi chạy: **Runtime → Change runtime type → T4 GPU**.

| Mục | Nội dung |
|---|---|
| Input | Fashion-MNIST (tải tự động bằng torchvision), `config.yaml` |
| Output | `config.yaml`, `common.py`, `results/baseline.csv` |
| Purpose | Chuẩn hóa dữ liệu, split train/val/test, hàm huấn luyện và baseline cho SV2, SV3 dùng chung (so sánh công bằng) |

In [1]:
# Cell 1: cài đặt, kiểm tra GPU, config chung
!pip -q install optuna pyyaml
import os, torch
os.makedirs('results', exist_ok=True)
print('GPU:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(đang chạy CPU - hãy bật GPU)')

config = '''seed: 42
n_train: 10000
n_val: 5000
epochs: 5
budget: 32
n_seeds: 10
search_space:
  lr: [1.0e-4, 1.0e-1]
  weight_decay: [1.0e-6, 1.0e-2]
  hidden: [32, 1024]
  dropout: [0.0, 0.5]
  batch_size: [64, 128, 256]
grid:
  lr: [1.0e-3, 1.0e-2]
  weight_decay: [1.0e-5, 1.0e-3]
  hidden: [64, 512]
  dropout: [0.0, 0.3]
  batch_size: [64, 256]
'''
open('config.yaml', 'w').write(config)
print(config)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 26.2 MB/s eta 0:00:00
GPU: True Tesla T4
seed: 42
n_train: 10000
n_val: 5000
epochs: 5
budget: 32
n_seeds: 10
search_space:
  lr: [1.0e-4, 1.0e-1]
  weight_decay: [1.0e-6, 1.0e-2]
  hidden: [32, 1024]
  dropout: [0.0, 0.5]
  batch_size: [64, 128, 256]
grid:
  lr: [1.0e-3, 1.0e-2]
  weight_decay: [1.0e-5, 1.0e-3]
  hidden: [64, 512]
  dropout: [0.0, 0.3]
  batch_size: [64, 256]



In [2]:
%%writefile common.py
# Module dùng chung: SV2, SV3 chỉ cần `from common import *`
import copy, time, numpy as np, torch, torch.nn as nn, yaml
from torchvision import datasets

CFG = yaml.safe_load(open('config.yaml'))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Baseline: cấu hình mặc định hợp lý, không tuning
BASELINE = dict(lr=1e-3, weight_decay=0.0, hidden=128, dropout=0.0, batch_size=128)

def load_data():
    # Tải Fashion-MNIST; chia cố định train/val từ tập train gốc, test = tập test gốc
    tr = datasets.FashionMNIST('/content/data', train=True, download=True)
    te = datasets.FashionMNIST('/content/data', train=False, download=True)
    prep = lambda d: ((d.data.float() / 255 - 0.2860) / 0.3530).reshape(len(d.data), -1)
    X, y, Xte, yte = prep(tr), tr.targets, prep(te), te.targets
    idx = np.random.RandomState(CFG['seed']).permutation(len(X))
    a, b = CFG['n_train'], CFG['n_train'] + CFG['n_val']
    d = dict(Xtr=X[idx[:a]], ytr=y[idx[:a]], Xva=X[idx[a:b]], yva=y[idx[a:b]], Xte=Xte, yte=yte)
    return {k: v.to(DEVICE) for k, v in d.items()}

def make_mlp(hidden, dropout):
    return nn.Sequential(
        nn.Linear(784, hidden), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(hidden, 10)).to(DEVICE)

def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        return (model(X).argmax(1) == y).float().mean().item()

def latency_ms(model, X, n_rep=30):
    # Độ trễ suy diễn trên CPU: median thời gian / mẫu (ms), batch 256
    m = copy.deepcopy(model).cpu().eval()
    xb, ts = X[:256].cpu(), []
    with torch.no_grad():
        for _ in range(3):
            m(xb)
        for _ in range(n_rep):
            t = time.perf_counter(); m(xb); ts.append(time.perf_counter() - t)
    return float(np.median(ts)) / len(xb) * 1000

def train_eval(params, data, train_seed=None, epochs=None):
    # f(x): huấn luyện MLP với siêu tham số x, trả về metric trên tập VALIDATION
    seed = CFG['seed'] if train_seed is None else train_seed
    epochs = epochs or CFG['epochs']
    torch.manual_seed(seed)
    model = make_mlp(int(params['hidden']), float(params['dropout']))
    opt = torch.optim.Adam(model.parameters(), lr=float(params['lr']),
                           weight_decay=float(params['weight_decay']))
    lossf, bs = nn.CrossEntropyLoss(), int(params['batch_size'])
    X, y = data['Xtr'], data['ytr']
    t0 = time.perf_counter()
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(len(X), device=DEVICE)
        for i in range(0, len(X), bs):
            b = perm[i:i + bs]
            opt.zero_grad()
            lossf(model(X[b]), y[b]).backward()
            opt.step()
    train_time = time.perf_counter() - t0
    metrics = dict(val_acc=accuracy(model, data['Xva'], data['yva']),
                   n_params=sum(p.numel() for p in model.parameters()),
                   latency_ms=latency_ms(model, data['Xva']),
                   train_time=train_time)
    return metrics, model

def test_accuracy(model, data):
    # CHỈ gọi một lần ở bước báo cáo cuối, không dùng để chọn tham số
    return accuracy(model, data['Xte'], data['yte'])

Writing common.py


In [3]:
# Cell 3: tải dữ liệu, kiểm tra và ước lượng thời gian
from common import *
data = load_data()
print({k: tuple(v.shape) for k, v in data.items()})
print('Số mẫu mỗi lớp (train):', torch.bincount(data['ytr']).tolist())

m, _ = train_eval(BASELINE, data)
print('1 lần đánh giá baseline:', {k: round(v, 4) for k, v in m.items()})
n_trials = 10 * CFG['budget'] * 2 + CFG['budget']   # Random + TPE (10 seed) + Grid
print(f'Ước tính tổng thời gian thí nghiệm chính: ~{n_trials * m["train_time"] / 60:.0f} phút ({n_trials} trial)')

100%|██████████| 26.4M/26.4M [00:00<00:00, 114MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 4.00MB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 61.4MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 13.4MB/s]


{'Xtr': (10000, 784), 'ytr': (10000,), 'Xva': (5000, 784), 'yva': (5000,), 'Xte': (10000, 784), 'yte': (10000,)}
Số mẫu mỗi lớp (train): [1009, 1002, 997, 1008, 997, 1015, 1016, 947, 1001, 1008]
1 lần đánh giá baseline: {'val_acc': 0.8394, 'n_params': 118282, 'latency_ms': 0.003, 'train_time': 1.1164}
Ước tính tổng thời gian thí nghiệm chính: ~13 phút (672 trial)


In [4]:
# Cell 4: Baseline (tham số mặc định) chạy 10 seed huấn luyện, báo cáo mean ± std
import pandas as pd
rows = []
for s in range(CFG['n_seeds']):
    m, model = train_eval(BASELINE, data, train_seed=s)
    m['test_acc'] = test_accuracy(model, data)   # test chỉ để báo cáo
    m['seed'] = s
    rows.append(m)

base = pd.DataFrame(rows)
base.to_csv('results/baseline.csv', index=False)
print(base[['val_acc', 'test_acc', 'latency_ms', 'n_params', 'train_time']].agg(['mean', 'std']).round(4))
base

      val_acc  test_acc  latency_ms  n_params  train_time
mean   0.8575    0.8423      0.0033  118282.0      0.5733
std    0.0051    0.0043      0.0006       0.0      0.0099


,val_acc,n_params,latency_ms,train_time,test_acc,seed
0,0.8640,118282,0.002923,0.577273,0.8463,0
1,0.8642,118282,0.004926,0.579742,0.8465,1
2,0.8586,118282,0.003146,0.564250,0.8364,2
3,0.8532,118282,0.003061,0.578077,0.8443,3
4,0.8616,118282,0.002895,0.566040,0.8466,4
5,0.8512,118282,0.003081,0.571383,0.8395,5
6,0.8530,118282,0.003028,0.562129,0.8460,6
7,0.8534,118282,0.003199,0.583151,0.8393,7
8,0.8624,118282,0.002954,0.590382,0.8428,8
9,0.8538,118282,0.003337,0.560950,0.8354,9


In [5]:
# Cell 5: sao lưu lên Google Drive (phiên Colab có thể bị ngắt)
from google.colab import drive
import shutil
drive.mount('/content/drive')
dst = '/content/drive/MyDrive/Nhom08_HPO'
os.makedirs(dst, exist_ok=True)
for f in ['config.yaml', 'common.py']:
    shutil.copy(f, dst)
shutil.copytree('results', dst + '/results', dirs_exist_ok=True)
print('Đã lưu vào', dst)

Mounted at /content/drive
Đã lưu vào /content/drive/MyDrive/Nhom08_HPO
